# 🎨 Rembg AI — Google Colab (A100 GPU)

---

> **📋 Bu notebook hakkında:**  
> Rembg kütüphanesini kullanarak yapay zeka ile arka plan kaldırma işlemi yapar.  
> Tüm ayarlar **form arayüzü** üzerinden yapılır — kod görmene gerek yok.  
> Sonuçlar doğrudan **Google Drive**'a kaydedilir.

---

## 🗺️ Notebook Haritası

| Blok | Ne Yapar? | Çalıştırma |
|------|-----------|------------|
| **🔧 BLOK 1** | Paket kurulumu (versiyon sabitli) | Sadece ilk seferde |
| **💾 BLOK 2** | Google Drive bağlantısı | Her oturumda |
| **🖥️ BLOK 3** | GPU kontrolü & kütüphane yükleme | Her oturumda |
| **⚙️ BLOK 4** | Ayarlar formu (model, kalite, yollar) | Her oturumda |
| **🖼️ BLOK 5** | Tekli resim işleme | İsteğe bağlı |
| **📦 BLOK 6** | Toplu işlem (klasör) | İsteğe bağlı |
| **📊 BLOK 7** | Sonuç raporu | İşlem sonrası |

---

⚠️ **ÖNEMLİ:** Üst menüden `Çalışma Zamanı → GPU türünü değiştir → A100` seçili olduğundan emin ol!

---
## 🔧 BLOK 1 — Paket Kurulumu

> **Ne yapar?**  
> - `rembg` ve tüm bağımlılıklarını **sabit sürümlerle** kurar  
> - `onnxruntime-gpu` kurar → A100 GPU'yu tam performansta kullanır  
> - Sabit sürümler sayesinde gelecekteki güncellemeler bu notebook'u **bozmaz**  
> 
> ⏱️ İlk kurulum ~2-3 dakika sürer. Sonraki oturumlarda tekrar çalıştır.

In [ ]:
#@title 🔧 BLOK 1 — Paket Kurulumu { display-mode: "form" }
#@markdown > Sabit kütüphane sürümlerini yükleyerek olası çakışmaları engeller. A100 GPU optimizasyonlarını etkinleştirir.

import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ HATA: {result.stderr[-300:]}')
    return result.returncode == 0

print('📦 Paketler kuruluyor...')

# 1. CUDA uyumlu onnxruntime-gpu (A100 = CUDA 12.x)
print('  ⚡ onnxruntime-gpu...')
run('pip install -q onnxruntime-gpu==1.20.1')

# 2. rembg core (sabit sürüm)
print('  🎨 rembg...')
run('pip install -q "rembg==2.0.59"')

# 3. ipywidgets & tqdm & pillow
print('  🎛️ ipywidgets & tqdm...')
run('pip install -q ipywidgets==8.1.5 tqdm pillow')

print('✅ Kurulum tamamlandı!')


---
## 💾 BLOK 2 — Google Drive Bağlantısı

> **Ne yapar?**  
> - Google Drive'ını Colab'a bağlar  
> - Tüm işlenmiş resimler **Drive'ına otomatik kaydedilir**  
> - `/content/drive/MyDrive/` altına kayıt yapılır  
> - İşlenmiş dosyalar kalıcıdır — oturum kapansa bile kaybolmaz  
> 
> 🔐 Google hesabı ile yetkilendirme isteyecek — izin ver.

In [ ]:
#@title 💾 BLOK 2 — Google Drive Bağlantısı { display-mode: "form" }
#@markdown > İşlenen görsellerin kalıcı olarak Drive hesabınıza kaydedilmesi için bu adımı çalıştırın.

from google.colab import drive
import os, subprocess

drive.mount('/content/drive', force_remount=True)

# Drive bağlantısını doğrula
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Google Drive başarıyla bağlandı!')
    print(f'📁 Drive yolu: /content/drive/MyDrive')
    
    # Drive boyutunu kontrol et
    result = subprocess.run(
        'df -h /content/drive/MyDrive 2>/dev/null | tail -1',
        shell=True, capture_output=True, text=True
    )
    if result.stdout:
        parts = result.stdout.split()
        if len(parts) >= 4:
            print(f'💾 Drive: {parts[1]} toplam, {parts[2]} kullanılıyor, {parts[3]} boş')
else:
    print('❌ Drive bağlanamadı! Lütfen tekrar çalıştır.')


---
## 🖥️ BLOK 3 — GPU Kontrolü & Kütüphaneler

> **Ne yapar?**  
> - Hangi GPU'nun aktif olduğunu kontrol eder  
> - `onnxruntime-gpu`'nun CUDA'yı görüp görmediğini doğrular  
> - A100 görünüyorsa GPU ile işlem yapılır (CPU'dan ~10-20x hızlı!)  
> 
> ⚠️ GPU görünmüyorsa: `Çalışma Zamanı → GPU türünü değiştir → A100`

In [ ]:
#@title 🖥️ BLOK 3 — GPU Kontrolü & Kütüphane Yükleme { display-mode: "form" }
#@markdown > A100 GPU'nun aktif olup olmadığını ve kütüphanelerin düzgün yüklendiğini kontrol eder.

import os, io, json, time, shutil, hashlib, subprocess
from pathlib import Path
from datetime import datetime
from typing import Optional, Tuple

from PIL import Image
import numpy as np
from tqdm.notebook import tqdm

# ── GPU Bilgisi ──────────────────────────────────────────────────
print('=' * 55)
print('  🖥️  GPU DURUM RAPORU')
print('=' * 55)

gpu_info = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True)
if gpu_info.returncode == 0:
    lines = gpu_info.stdout.split('\n')
    for line in lines:
        if any(x in line for x in ['CUDA', 'A100', 'V100', 'T4', 'Driver', 'MiB', 'GPU']):
            print(f'  {line.strip()}')
else:
    print('  ⚠️  nvidia-smi bulunamadı — GPU yok olabilir')
    print('  ⚠️  Çalışma Zamanı → GPU türünü değiştir → A100')

print()

# ── ONNX Runtime GPU Kontrolü ────────────────────────────────────
try:
    import onnxruntime as ort
    providers = ort.get_available_providers()
    print(f'  🔧 ONNX Runtime: v{ort.__version__}')
    print(f'  📡 Mevcut sağlayıcılar: {providers}')
    if 'CUDAExecutionProvider' in providers:
        print('  ✅ GPU (CUDA) aktif → A100 kullanılıyor!')
        USE_GPU = True
    else:
        print('  ⚠️  GPU bulunamadı → CPU modunda çalışılacak')
        USE_GPU = False
except Exception as e:
    print(f'  ❌ ONNX Runtime hatası: {e}')
    USE_GPU = False

print()

# ── rembg Import ─────────────────────────────────────────────────
try:
    from rembg import remove, new_session
    import rembg
    print(f'  ✅ rembg: v{rembg.__version__}')
except Exception as e:
    print(f'  ❌ rembg import hatası: {e}')
    print('  💡 BLOK 1 hücrelerini tekrar çalıştır!')

print('=' * 55)
print(f'  Mod: {"🚀 GPU (A100)" if USE_GPU else "🐌 CPU (yavaş)"}')
print('=' * 55)


---
## ⚙️ BLOK 4 — Ayarlar Formu

> **Ne yapar?**  
> - Tüm parametreleri görsel formda ayarlarsın  
> - **Model seçimi:** Kullanım amacına göre AI modeli  
> - **Google Drive yolları:** Giriş ve çıkış klasörleri  
> - **Alpha Matting:** Saç, kürk, ince kenarlıklar için iyileştirme  
> - **Skip modu:** Zaten işlenmiş dosyaları atlama  
> 
> 📝 Formu doldurduktan sonra **"💾 Ayarları Kaydet"** butonuna bas!

### 🧠 Model Rehberi

| Model | Tür | Kullanım Alanı | Hız | Kalite |
|-------|-----|----------------|-----|--------|
| `u2net` | 🎯 Genel | Genel amaçlı | 🟡 Orta | ⭐⭐⭐⭐ |
| `u2netp` | ⚡ Hızlı | Toplu/seri işlem | 🟢 Hızlı | ⭐⭐⭐ |
| `u2net_human_seg` | 👤 İnsan | Portre, insan fotoğrafı | 🟡 Orta | ⭐⭐⭐⭐ |
| `u2net_cloth_seg` | 👗 Kıyafet | E-ticaret, tekstil | 🟡 Orta | ⭐⭐⭐⭐ |
| `silueta` | 🖤 Siluet | Sanatsal siluet | 🟢 Hızlı | ⭐⭐⭐ |
| `birefnet-general` | ✨ BiRefNet | **Profesyonel genel** | 🔴 Yavaş | ⭐⭐⭐⭐⭐ |
| `birefnet-portrait` | 🧑 Portre | **Headshot, profil** | 🔴 Yavaş | ⭐⭐⭐⭐⭐ |
| `dis-general-use` | 🌐 DIS | İnce kenarlık, saç, kürk | 🟡 Orta | ⭐⭐⭐⭐⭐ |
| `dis-anime` | 🎌 Anime | Anime, çizgi karakter | 🟡 Orta | ⭐⭐⭐⭐⭐ |
| `bria-rmbg` | 🏢 Ticari | Ürün fotoğrafçılığı | 🟡 Orta | ⭐⭐⭐⭐⭐ |

In [ ]:
#@title ⚙️ BLOK 4 — Ayarlar Formu { display-mode: "form" }

#@markdown ### 🧠 Model Seçimi
#@markdown * **u2net**: Genel amaçlı
#@markdown * **u2netp**: En hızlı, düşük kalite
#@markdown * **u2net_human_seg**: İnsan/portre
#@markdown * **u2net_cloth_seg**: Kıyafet/tekstil
#@markdown * **silueta**: Sanatsal siluet
#@markdown * **birefnet-***: Yeni nesil, en yüksek kaliteli modeller
#@markdown * **dis-general-use**: İnce detaylar (saç, kürk)
#@markdown * **bria-rmbg**: Ticari/ürün fotoğrafları
model = "u2net" #@param ["u2net", "u2netp", "u2net_human_seg", "u2net_cloth_seg", "silueta", "birefnet-general", "birefnet-general-lite", "birefnet-portrait", "birefnet-dis", "birefnet-hrsod", "birefnet-cod", "birefnet-massive", "dis-general-use", "dis-anime", "bria-rmbg"]

#@markdown ### 📁 Google Drive Yolları
input_path = "/content/drive/MyDrive/rembg_input" #@param {type:"string"}
output_path = "/content/drive/MyDrive/rembg_output" #@param {type:"string"}
output_suffix = "_rembg" #@param {type:"string"}

#@markdown ### 🔬 Alpha Matting (Kenarlık İyileştirme)
alpha = False #@param {type:"boolean"}
fg = 240 #@param {type:"slider", min:0, max:255, step:1}
bg = 10 #@param {type:"slider", min:0, max:255, step:1}
erode = 10 #@param {type:"slider", min:0, max:50, step:1}

#@markdown ### 🎨 Ek Seçenekler
only_mask = False #@param {type:"boolean"}
ppm = True #@param {type:"boolean"}
skip = True #@param {type:"boolean"}

use_bgcolor = False #@param {type:"boolean"}
bgcolor_hex = "#ffffff" #@param {type:"color"}
bgcolor_alpha = 255 #@param {type:"slider", min:0, max:255, step:1}

# ── Ayarları Kaydet ve Klasörleri Oluştur ───────────────────────
from pathlib import Path
import shutil

CONFIG = {
    'model': model,
    'input_path': input_path,
    'output_path': output_path,
    'output_suffix': output_suffix,
    'alpha': alpha,
    'fg': fg,
    'bg': bg,
    'erode': erode,
    'only_mask': only_mask,
    'ppm': ppm,
    'skip': skip,
    'use_bgcolor': use_bgcolor,
    'bgcolor_hex': bgcolor_hex,
    'bgcolor_alpha': bgcolor_alpha,
}

# Klasörleri oluştur
Path(CONFIG['output_path']).mkdir(parents=True, exist_ok=True)
inp = Path(CONFIG['input_path'])
if not inp.exists():
    inp.mkdir(parents=True, exist_ok=True)
    print(f'📁 Giriş klasörü oluşturuldu: {CONFIG["input_path"]}')
    print('   Görsellerinizi bu klasöre yükleyin.')
else:
    exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.gif')
    imgs = [f for f in inp.iterdir() if f.suffix.lower() in exts]
    print(f'📁 Giriş klasörü: {CONFIG["input_path"]} ({len(imgs)} görsel bulundu)')

print(f'📁 Çıkış klasörü: {CONFIG["output_path"]}')

# ── OTOMATİK MODEL CACHE KONTROLÜ ─────────────────────────────────
def setup_model_cache():
    home_cache = Path.home() / '.u2net'
    home_cache.mkdir(parents=True, exist_ok=True)
    
    drive_cache = Path('/content/drive/MyDrive/.u2net_model_cache')
    if drive_cache.exists():
        copied_any = False
        for item in drive_cache.glob('*.onnx'):
            dest = home_cache / item.name
            if not dest.exists():
                print(f"  ⬇️ Drive'dan model yüklendi: {item.name}")
                shutil.copy2(item, dest)
                copied_any = True
        if copied_any:
            print("✅ Önbellekteki modeller Google Drive'dan başarıyla yerel diske kopyalandı!")
        else:
            print("ℹ️ Modeller zaten yerel diskte mevcut veya önbellek boş.")
    else:
        print("ℹ️ Google Drive önbellek klasörü bulunamadı. İlk işlemde modeller indirilip otomatik olarak Drive'a yedeklenecektir.")

def backup_model_cache():
    home_cache = Path.home() / '.u2net'
    drive_cache = Path('/content/drive/MyDrive/.u2net_model_cache')
    drive_cache.mkdir(parents=True, exist_ok=True)
    
    if home_cache.exists():
        backed_up = False
        for item in home_cache.glob('*.onnx'):
            dest = drive_cache / item.name
            if not dest.exists():
                print(f"  💾 Google Drive'a yedekleniyor: {item.name}")
                shutil.copy2(item, dest)
                backed_up = True
        if backed_up:
            print("✅ Yeni indirilen modeller Google Drive önbelleğine kaydedildi!")

setup_model_cache()
print('\n✅ Ayarlar kaydedildi ve doğrulandı!')


---
## 🖼️ BLOK 5 — Tekli Resim İşleme

> **Ne yapar?**  
> - Bilgisayarından **tek bir resim** yükler  
> - Seçtiğin model ile arka planı kaldırır  
> - Sonucu hem **ekranda** gösterir hem **Drive'a** kaydeder  
> 
> 💡 İlk kez çalışırken model indirilir (~50MB-1GB arası), biraz beklemeni gerekebilir.  
> 💡 İkinci çalışmada model cache'de → anında başlar.

---

⚠️ **Önce BLOK 4'teki "Ayarları Kaydet" butonuna basmayı unutma!**

In [ ]:
#@title 🖼️ BLOK 5 — Tekli Resim İşleme { display-mode: "form" }
#@markdown > Bilgisayarınızdan tek bir görsel yükleyerek işlem yapın. Sonuç karşılaştırmalı olarak gösterilir ve Drive'a kaydedilir.

from google.colab import files as colab_files
import io
from PIL import Image
from IPython.display import display, HTML, Image as IPImage
import time

def hex_to_rgba(hex_color: str, alpha: int = 255):
    h = hex_color.lstrip('#')
    if len(h) == 6:
        return (int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16), alpha)
    return None

def get_output_path(input_path: Path, suffix: str, out_dir: str) -> Path:
    return Path(out_dir) / f'{input_path.stem}{suffix}.png'

def process_image(img_bytes: bytes, cfg: dict) -> bytes:
    session = new_session(cfg['model'])
    bgcolor = None
    if cfg.get('use_bgcolor') and cfg.get('bgcolor_hex'):
        bgcolor = hex_to_rgba(cfg['bgcolor_hex'], cfg.get('bgcolor_alpha', 255))
    return remove(
        img_bytes,
        session=session,
        alpha_matting=cfg['alpha'],
        alpha_matting_foreground_threshold=cfg['fg'],
        alpha_matting_background_threshold=cfg['bg'],
        alpha_matting_erode_size=cfg['erode'],
        only_mask=cfg['only_mask'],
        post_process_mask=cfg['ppm'],
        bgcolor=bgcolor,
    )

def display_comparison(orig_bytes: bytes, result_bytes: bytes, filename: str):
    orig_img = Image.open(io.BytesIO(orig_bytes)).convert('RGBA')
    res_img  = Image.open(io.BytesIO(result_bytes)).convert('RGBA')

    max_size = 512
    orig_img.thumbnail((max_size, max_size))
    res_img.thumbnail((max_size, max_size))

    checker = Image.new('RGBA', res_img.size)
    cw, ch = 20, 20
    for y in range(0, ch * 20, ch):
        for x in range(0, cw * 20, cw):
            c = (200, 200, 200, 255) if ((x // cw + y // ch) % 2 == 0) else (255, 255, 255, 255)
            for py in range(ch):
                for px in range(cw):
                    if x + px < res_img.width and y + py < res_img.height:
                        checker.putpixel((x + px, y + py), c)
    bg = Image.new('RGBA', res_img.size)
    bg.paste(checker, (0, 0))
    result_with_bg = Image.alpha_composite(bg, res_img)

    combined_w = orig_img.width + result_with_bg.width + 20
    combined_h = max(orig_img.height, result_with_bg.height) + 40
    combined = Image.new('RGBA', (combined_w, combined_h), (248, 249, 250, 255))
    combined.paste(orig_img.convert('RGBA'), (0, 30))
    combined.paste(result_with_bg, (orig_img.width + 20, 30))

    buf = io.BytesIO()
    combined.convert('RGB').save(buf, format='PNG')
    display(HTML(f'<p style="font-weight:600;color:#553c9a">📸 {filename} — Orijinal (sol) | Sonuç (sağ)</p>'))
    display(IPImage(data=buf.getvalue()))

if 'CONFIG' not in globals():
    print('⚠️ Lütfen önce BLOK 4 (Ayarlar Formu) hücresini çalıştırın!')
else: 
    uploaded = colab_files.upload()
    if not uploaded:
        print('❌ Dosya seçilmedi.')
    else:
        for filename, content in uploaded.items():
            print(f'\n🔄 İşleniyor: {filename} ({len(content)/1024/1024:.1f} MB)...')
            t0 = time.time()
            try:
                setup_model_cache()
                result_bytes = process_image(content, CONFIG)
                elapsed = time.time() - t0
                
                out_path = get_output_path(Path(filename), CONFIG['output_suffix'], CONFIG['output_path'])
                with open(out_path, 'wb') as f:
                    f.write(result_bytes)
                
                backup_model_cache()

                print(f'✅ Tamamlandı! ({elapsed:.1f}s)')
                print(f'💾 Kaydedildi: {out_path}')
                print(f'📦 Dosya boyutu: {len(result_bytes)/1024/1024:.2f} MB\n')
                display_comparison(content, result_bytes, filename)
            except Exception as e:
                import traceback
                print(f'❌ Hata: {e}')
                traceback.print_exc()


---
## 📦 BLOK 6 — Toplu İşlem (Batch)

> **Ne yapar?**  
> - Drive'daki giriş klasöründeki **tüm resimleri** işler  
> - Zaten işlenmiş dosyaları **skip** eder (atlar) — güvenli devam  
> - Gerçek zamanlı **progress bar** gösterir  
> - İşlem sonunda **özet rapor** verir  
> - GB boyutunda dosyaları bile sorunsuz işler  

> ### 🔄 Skip (Atlama) Mantığı
> Skip aktifse: Çıkış klasöründe `dosya_adi_rembg.png` zaten varsa **tekrar işlenmez**.  
> Bu sayede:
> - İşlem yarıda kesilirse → kaldığı yerden devam eder  
> - Colab oturumu kapanırsa → tekrar çalıştır, sadece eksikler işlenir  
> - Aynı klasöre yeni resim eklenirse → sadece yeniler işlenir  

> ⚠️ **Giriş klasörüne resim eklemek için:**  
> Drive web arayüzünden (`drive.google.com`) yükleyebilir veya aşağıdaki yükleme hücresini kullanabilirsin.

In [ ]:
#@title 📤 BLOK 6A — Drive Giriş Klasörüne Toplu Resim Yükleme { display-mode: "form" }
#@markdown > Bilgisayarınızdan seçeceğiniz birden fazla görseli doğrudan Google Drive'daki giriş klasörüne yükler.

from google.colab import files as colab_files
from pathlib import Path

if 'CONFIG' not in globals():
    print('⚠️ Lütfen önce BLOK 4 (Ayarlar Formu) hücresini çalıştırın!')
else:
    inp_dir = Path(CONFIG['input_path'])
    inp_dir.mkdir(parents=True, exist_ok=True)
    print(f'📁 Yükleme hedefi: {inp_dir}')
    
    uploaded = colab_files.upload()
    if not uploaded:
        print('❌ Dosya seçilmedi.')
    else:
        for filename, content in uploaded.items():
            dest = inp_dir / filename
            with open(dest, 'wb') as f:
                f.write(content)
            print(f'  ✅ {filename} → Drive ({len(content)/1024/1024:.1f} MB)')
            
        exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.gif')
        total = len([f for f in inp_dir.iterdir() if f.suffix.lower() in exts])
        print(f'\n📊 Giriş klasöründe toplam {total} resim hazır. Şimdi toplu işlem hücresini çalıştırabilirsiniz.')


In [ ]:
#@title 🚀 BLOK 6B — Toplu İşlemi Başlat { display-mode: "form" }
#@markdown > Google Drive giriş klasöründeki tüm görselleri toplu olarak işler.
#@markdown > **Durdurmak için** yukarıdaki durdur (stop/interrupt execution) düğmesine basabilirsiniz.

import time
from pathlib import Path

STATS = {'processed': 0, 'skipped': 0, 'failed': 0, 'total_mb': 0.0, 'start_time': None, 'results': []}

if 'CONFIG' not in globals():
    print('⚠️ Lütfen önce BLOK 4 (Ayarlar Formu) hücresini çalıştırın!')
else:
    inp_dir = Path(CONFIG['input_path'])
    out_dir = Path(CONFIG['output_path'])
    out_dir.mkdir(parents=True, exist_ok=True)

    EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif', '.gif'}
    if not inp_dir.exists():
        print(f'❌ Giriş klasörü bulunamadı: {inp_dir}')
    else:
        all_files = sorted([f for f in inp_dir.iterdir() if f.suffix.lower() in EXTS])
        if not all_files:
            print(f'❌ Giriş klasöründe resim bulunamadı: {inp_dir}')
            print(f'   Desteklenen formatlar: { ", ".join(EXTS) }')
        else:
            print('=' * 60)
            print('  📦 TOPLU İŞLEM BAŞLIYOR')
            print('=' * 60)
            print(f'  📁 Giriş : {inp_dir}')
            print(f'  📁 Çıkış : {out_dir}')
            print(f'  🧠 Model : {CONFIG["model"]}')
            print(f'  🔬 Alpha : {"Aktif" if CONFIG["alpha"] else "Kapalı"}')
            print(f'  ⏭️  Skip  : {"Aktif" if CONFIG["skip"] else "Kapalı"}')
            print(f'  🖥️  GPU   : {"A100" if USE_GPU else "CPU"}')
            print(f'  📊 Toplam: {len(all_files)} dosya')
            print('=' * 60)
            print()

            setup_model_cache()
            print(f'🔄 Model yükleniyor: {CONFIG["model"]}...')
            session = new_session(CONFIG['model'])
            backup_model_cache()
            print('✅ Model hazır!\n')

            STATS.update({'processed': 0, 'skipped': 0, 'failed': 0, 'total_mb': 0.0, 'start_time': time.time(), 'results': []})
            
            bgcolor = None
            if CONFIG.get('use_bgcolor') and CONFIG.get('bgcolor_hex'):
                bgcolor = hex_to_rgba(CONFIG['bgcolor_hex'], CONFIG.get('bgcolor_alpha', 255))

            pbar = tqdm(all_files, desc='Resimler işleniyor', unit='resim', bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

            try:
                for img_path in pbar:
                    out_path = out_dir / f'{img_path.stem}{CONFIG["output_suffix"]}.png'

                    if CONFIG['skip'] and out_path.exists():
                        pbar.set_postfix({'durum': '⏭️ atlandı'})
                        STATS['skipped'] += 1
                        STATS['results'].append({'file': img_path.name, 'status': 'skipped'})
                        continue

                    try:
                        file_size_mb = img_path.stat().st_size / 1024 / 1024
                        pbar.set_postfix({'dosya': img_path.name[:20], 'boyut': f'{file_size_mb:.1f}MB'})

                        t0 = time.time()
                        with open(img_path, 'rb') as f:
                            img_bytes = f.read()

                        result_bytes = remove(
                            img_bytes,
                            session=session,
                            alpha_matting=CONFIG['alpha'],
                            alpha_matting_foreground_threshold=CONFIG['fg'],
                            alpha_matting_background_threshold=CONFIG['bg'],
                            alpha_matting_erode_size=CONFIG['erode'],
                            only_mask=CONFIG['only_mask'],
                            post_process_mask=CONFIG['ppm'],
                            bgcolor=bgcolor,
                        )

                        with open(out_path, 'wb') as f:
                            f.write(result_bytes)

                        elapsed = time.time() - t0
                        out_mb = len(result_bytes) / 1024 / 1024
                        STATS['processed'] += 1
                        STATS['total_mb'] += out_mb
                        STATS['results'].append({
                            'file': img_path.name,
                            'status': 'ok',
                            'elapsed': elapsed,
                            'out_mb': out_mb,
                        })
                        pbar.set_postfix({'durum': f'✅ {elapsed:.1f}s'})
                    except Exception as e:
                        STATS['failed'] += 1
                        STATS['results'].append({'file': img_path.name, 'status': 'error', 'error': str(e)})
                        pbar.set_postfix({'durum': '❌ hata'})
                        print(f'\n  ❌ Hata — {img_path.name}: {e}')
            except KeyboardInterrupt:
                print('\n⏹️ İşlem kullanıcı tarafından durduruldu!')

            pbar.close()

            total_time = time.time() - STATS['start_time']
            print()
            print('=' * 60)
            print('  📊 İŞLEM ÖZETİ')
            print('=' * 60)
            print(f'  ✅ İşlendi  : {STATS["processed"]} dosya')
            print(f'  ⏭️  Atlandı  : {STATS["skipped"]} dosya')
            print(f'  ❌ Hata     : {STATS["failed"]} dosya')
            print(f'  📦 Çıkış    : {STATS["total_mb"]:.1f} MB toplam')
            print(f'  ⏱️  Süre     : {total_time:.1f}s ({total_time/60:.1f} dk)')
            if STATS['processed'] > 0:
                print(f'  ⚡ Hız      : {STATS["processed"]/total_time:.2f} resim/sn')
            print(f'  📁 Konum    : {out_dir}')
            print('=' * 60)


---
## 📊 BLOK 7 — Sonuç Raporu & Özet

> **Ne yapar?**  
> - Toplu işlem sonrasında detaylı özet gösterir  
> - Başarılı/atlanan/hatalı dosyaları listeler  
> - Çıkış klasöründeki dosya sayısını ve toplam boyutu gösterir  
> - Raporu JSON olarak Drive'a kaydeder (opsiyonel)

---

In [ ]:
#@title 📊 BLOK 7 — Sonuç Raporu & Çıkış Klasörü Durumu { display-mode: "form" }
#@markdown > Çıkış klasöründeki dosyaların boyutlarını ve en son işlenen görsellerin raporunu görüntüler.

if 'CONFIG' not in globals():
    print('⚠️ Lütfen önce BLOK 4 (Ayarlar Formu) hücresini çalıştırın!')
else:
    out_dir = Path(CONFIG['output_path'])
    print('=' * 60)
    print('  📊 ÇIKIŞ KLASÖRÜ RAPORU')
    print('=' * 60)

    if not out_dir.exists():
        print(f'  ❌ Çıkış klasörü henüz oluşturulmadı: {out_dir}')
    else:
        png_files = sorted(out_dir.glob('*.png'))
        total_size = sum(f.stat().st_size for f in png_files)

        print(f'  📁 Klasör  : {out_dir}')
        print(f'  📊 Dosya   : {len(png_files)} PNG')
        print(f'  💾 Toplam  : {total_size/1024/1024:.1f} MB ({total_size/1024/1024/1024:.2f} GB)')
        print()

        if 'STATS' in globals() and STATS.get('results'):
            print('  📋 Son Toplu İşlem Sonuçları:')
            print(f'     ✅ İşlendi  : {STATS["processed"]}')
            print(f'     ⏭️  Atlandı  : {STATS["skipped"]}')
            print(f'     ❌ Hata     : {STATS["failed"]}')
            if STATS.get('start_time') and STATS['start_time']:
                elapsed = time.time() - STATS['start_time']
                print(f'     ⏱️  Süre     : {elapsed:.0f}s')
            print()

        if png_files:
            print('  📋 Son 10 İşlenmiş Dosya:')
            for f in png_files[-10:]:
                size_kb = f.stat().st_size / 1024
                mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
                print(f'     • {f.name:<40} {size_kb:>8.0f} KB  {mtime}')
        print('=' * 60)


---
## 🛠️ BLOK 8 — Bakım & Yardım

> **Ne yapar?**  
> - Model cache temizleme  
> - Colab disk kullanım kontrolü  
> - Sorun giderme araçları  

### ❓ Sık Karşılaşılan Sorunlar

| Sorun | Çözüm |
|-------|-------|
| `No module named 'rembg'` | BLOK 1'i tekrar çalıştır |
| GPU görünmüyor | Çalışma zamanı → GPU türünü değiştir → A100 |
| Drive bağlanmıyor | BLOK 2'yi tekrar çalıştır |
| Model indirilmiyor | İnternet bağlantısını kontrol et |
| Colab oturumu kapandı | BLOK 2-3-4'ü çalıştır, sonra BLOK 6B'yi çalıştır (skip devam eder) |
| RAM yetersiz | Daha küçük batch'lerle çalış veya u2netp modelini seç |

In [ ]:
#@title ⚙️ BLOK 8 — Disk ve Cache Durumu { display-mode: "form" }
#@markdown > Sunucu disk kullanımını kontrol eder ve model dosyalarınızı listeler.

import subprocess
from pathlib import Path

print('💾 COLAB DISK KULLANIMI:')
r = subprocess.run('df -h / /content 2>/dev/null', shell=True, capture_output=True, text=True)
print(r.stdout)

cache_dir = Path.home() / '.u2net'
if cache_dir.exists():
    models = list(cache_dir.glob('*.onnx'))
    total_cache = sum(m.stat().st_size for m in models) / 1024 / 1024 / 1024
    print(f'\n🧠 MODEL CACHE: {cache_dir}')
    for m in models:
        size_mb = m.stat().st_size / 1024 / 1024
        print(f'   • {m.name:<40} {size_mb:>6.0f} MB')
    print(f'   Toplam: {total_cache:.2f} GB')
else:
    print('\n🧠 Henüz indirilmiş model bulunmuyor (ilk işlemde indirilecektir).')
